# Sample 03: 相対ポインタ & オフセットテーブル (`Offset`, `OffsetTable`)

バイナリファイルフォーマット（フォント、3Dモデル、アーカイブ等）で頻出する、ヘッダーからの相対オフセットポインタ、オフセットテーブル配列、および遅延バックパッチを学びます。

### 学べる内容
- `Offset[Target, Base.SELF]` による構造体先頭相対オフセットポインタの自動解決
- `restored.field.subfield` によるデリファレンス済みオブジェクトへの透過アクセス
- `OffsetTable` による動的ポインタ配列
- 文字列/Enumキーによる名前付き遅延オフセット (`Offset["key"]`) と一括解決
- 名前空間スコープ (`with writer.namespace(...)`) によるキー衝突防止

In [1]:
from enum import Enum

from binary_master import (
    Base,
    BinaryWriter,
    Offset,
    OffsetTable,
    UInt8,
    UInt16,
    UInt32,
    binary_struct,
    hexdump,
    read_struct,
)

## 1. 相対ポインタ (`Offset[Target, Base.SELF]`)

ヘッダー内に `Offset[ImageData, Base.SELF]` を定義しておくと、シリアライズ時にターゲットの `ImageData` が自動的に末尾に配置され、その相対オフセットが自動バックパッチされます。

In [2]:
@binary_struct
class ImageData:
    width: UInt16
    height: UInt16
    format: UInt8

@binary_struct
class FileHeader:
    magic: UInt32
    # 構造体先頭からの相対オフセット
    image_offset: Offset[ImageData, Base.SELF] = None

header = FileHeader(
    magic=0x494D4730,
    image_offset=ImageData(width=1920, height=1080, format=1)
)

data = header.to_bytes()
print(f"シリアライズ結果 ({len(data)} バイト):")
print(hexdump(data, annotate=True))

# 復元: image_offset 経由で ImageData のフィールドへ直接アクセス可能！
restored = FileHeader.from_bytes(data)
print(f"ヘッダー magic: 0x{restored.magic:08X}")
print(f"画像サイズ: {restored.image_offset.width} x {restored.image_offset.height}")
assert restored.image_offset.width == 1920

シリアライズ結果 (13 バイト):
Offset    00 01 02 03 04 05 06 07  08 09 0A 0B 0C 0D 0E 0F  |     ASCII      |
------------------------------------------------------------------------------
00000000  30 47 4d 49 08 00 00 00  80 07 38 04 01           |0GMI......8..   |
  [Total: 13 bytes (`0x000D`)]
ヘッダー magic: 0x494D4730
画像サイズ: 1920 x 1080

## 2. オフセットテーブル配列 (`OffsetTable`)

複数の子要素へのポインタを配列として保持します。

In [3]:
@binary_struct
class LeafItem:
    item_id: UInt16
    value: UInt32

@binary_struct
class RootContainer:
    count: UInt16
    items: OffsetTable[2, UInt32, Base.SELF]

container = RootContainer(
    count=2,
    items=[
        LeafItem(item_id=1, value=100),
        LeafItem(item_id=2, value=200),
    ]
)

writer = BinaryWriter()
writer.write_struct(container)
c_bytes = writer.to_bytes()
print(f"OffsetTable シリアライズ ({len(c_bytes)} バイト):")
print(hexdump(c_bytes, annotate=True))

restored_c = RootContainer.from_bytes(c_bytes)
print(f"オフセットテーブルの格納値: {[f'0x{o:04X}' for o in restored_c.items]}")
# 各オフセット位置から LeafItem を復元
item0 = read_struct(LeafItem, c_bytes[restored_c.items[0]:])
item1 = read_struct(LeafItem, c_bytes[restored_c.items[1]:])
print(f"Item 0 value: {item0.value}")
print(f"Item 1 value: {item1.value}")
assert item0.value == 100
assert item1.value == 200

OffsetTable シリアライズ (22 バイト):
Offset    00 01 02 03 04 05 06 07  08 09 0A 0B 0C 0D 0E 0F  |     ASCII      |
------------------------------------------------------------------------------
00000000  02 00 0a 00 00 00 10 00  00 00 01 00 64 00 00 00  |............d...|
00000010  02 00 c8 00 00 00                                 |......          |
  [Total: 22 bytes (`0x0016`)]
オフセットテーブルの格納値: ['0x000A', '0x0010']
Item 0 value: 100
Item 1 value: 200

## 3. キーベースの遅延オフセット解決 (`Offset["key"]`)

「ヘッダーを先に書いて、途中に任意のデータを挟み、後から目的のオフセット位置を確定させたい」場合、文字列キーや Enum キーでオフセット枠を宣言し、`writer.write_named_offset("key")` で解決します。

In [4]:
class SectionKey(Enum):
    PAYLOAD = "payload"

@binary_struct
class StreamHeader:
    magic: UInt16
    payload_ptr: Offset[SectionKey.PAYLOAD, ImageData]

writer = BinaryWriter()
writer.write_struct(StreamHeader(magic=0x55AA))
writer.write_string("可変長の中間メタデータ...") # 間にデータを挟む

# ここで payload_ptr の位置を確定・自動配置！
writer.write_named_offset(SectionKey.PAYLOAD, ImageData(width=800, height=600, format=2))

stream_data = writer.to_bytes()
print(f"遅延オフセット解決バイナリ ({len(stream_data)} バイト):")
print(hexdump(stream_data, annotate=True))

restored_s = StreamHeader.from_bytes(stream_data)
print(f"復元されたペイロード: {restored_s.payload_ptr.width} x {restored_s.payload_ptr.height}")
assert restored_s.payload_ptr.height == 600

遅延オフセット解決バイナリ (47 バイト):
Offset    00 01 02 03 04 05 06 07  08 09 0A 0B 0C 0D 0E 0F  |     ASCII      |
------------------------------------------------------------------------------
00000000  aa 55 2a 00 00 00 e5 8f  af e5 a4 89 e9 95 b7 e3  |.U*.............|
00000010  81 ae e4 b8 ad e9 96 93  e3 83 a1 e3 82 bf e3 83  |................|
00000020  87 e3 83 bc e3 82 bf 2e  2e 2e 20 03 58 02 02     |.......... .X.. |
  [Total: 47 bytes (`0x002F`)]
復元されたペイロード: 800 x 600

## 4. 名前空間スコープ (`with writer.namespace(...)`)

ループ内で同じ構造体（同じキー名 `"payload"`）を繰り返し書き出す場合、`with writer.namespace("chunk", auto_id=True):` でスコープ化することでキー衝突を完全に防ぐことができます。

In [5]:
@binary_struct
class ChunkHeader:
    chunk_id: UInt16
    payload_offset: Offset["payload"]

nw = BinaryWriter()

# 2つのチャンクを同じキー名 "payload" で独立に書き込む
for i in range(2):
    with nw.namespace("chunk", auto_id=True):
        nw.write_struct(ChunkHeader(chunk_id=101 + i))
        nw.write_string(f"metadata_{i}...")
        nw.write_named_offset("payload")
        nw.write_string(f"ACTUAL_BODY_{i}")

ndata = nw.to_bytes()
print(f"スコープ付き名前空間バイナリ ({len(ndata)} バイト):")
print(hexdump(ndata, annotate=True))
print("全オフセット機能の検証成功！")

スコープ付き名前空間バイナリ (64 バイト):
Offset    00 01 02 03 04 05 06 07  08 09 0A 0B 0C 0D 0E 0F  |     ASCII      |
------------------------------------------------------------------------------
00000000  65 00 13 00 00 00 6d 65  74 61 64 61 74 61 5f 30  |e.....metadata_0|
00000010  2e 2e 2e 41 43 54 55 41  4c 5f 42 4f 44 59 5f 30  |...ACTUAL_BODY_0|
00000020  66 00 33 00 00 00 6d 65  74 61 64 61 74 61 5f 31  |f.3...metadata_1|
00000030  2e 2e 2e 41 43 54 55 41  4c 5f 42 4f 44 59 5f 31  |...ACTUAL_BODY_1|
  [Total: 64 bytes (`0x0040`)]
全オフセット機能の検証成功！